# Train MuJoCo Playground's SO-101 PickCube for mjswan

MuJoCo Playground 1.0.1's `SO101PickCube`, the state-observation sibling of the task its
sim-to-real demo deploys, trained with Playground's own environment and PPO settings, then
exported as the ONNX policy an mjswan scene runs.

1. **Runtime → Change runtime type → T4 GPU**.
2. Add a Hugging Face **write** token as the Colab secret `HF_TOKEN` (the key icon on the
   left), with notebook access on.
3. Set `HF_REPO` below and run the cells in order. Every checkpoint is kept in that
   repository, so when a session ends, running the cells again resumes from the last one.
4. The result is in the same repository: `policy.onnx`, `contract.json` and `reference.npz`
   (see [`so101_pick_cube.py`](https://github.com/ttktjmt/mjswan/blob/main/examples/train/so101_pick_cube.py)).

In [ ]:
!nvidia-smi

In [ ]:
import os

from google.colab import userdata

# A Hugging Face model repository, "user/name", created public if it does not exist.
HF_REPO = ""
ITERATIONS = 1000
# The mjswan branch the training script is fetched from.
BRANCH = "main"

if not HF_REPO:
    raise ValueError("Set HF_REPO: it keeps what an ended session loses.")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
OUT = "/content/so101_pick_cube"

In [ ]:
!wget -q -O so101_pick_cube.py https://raw.githubusercontent.com/ttktjmt/mjswan/{BRANCH}/examples/train/so101_pick_cube.py
!python so101_pick_cube.py setup

Training curves while it runs (optional; this session's only).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$OUT/checkpoints"

Train. If the session ends, run the cells above again and then this one: it resumes from
the newest checkpoint in `HF_REPO`.

In [ ]:
!python so101_pick_cube.py train --out "{OUT}" --iterations {ITERATIONS} --hf-repo {HF_REPO}

In [ ]:
!python so101_pick_cube.py export --out "{OUT}" --hf-repo {HF_REPO}
!python so101_pick_cube.py upload --out "{OUT}" --hf-repo {HF_REPO}